# Lecture 03: Embeddings and PyTorch for Language Models

CS40008.01 · Baojian Zhou · Fudan University · September 23, 2026

**How do discrete tokens become trainable representations?**

Predict each result before running its cell. E01–E06 follow the slide order. These are **ungraded practices**, separate from Quiz 1 and Assignment A1. The complete notebook runs offline on CPU in the core `uv sync` environment; no model weights or tokenizer downloads are needed. The core examples and optional skip-gram corpus are **toy data**. Optional P01 uses a published co-occurrence table, cited there.

| Exercise | What you check |
| --- | --- |
| E01 · 4 min | Lookup shapes and one-hot equivalence |
| E02 · 4 min | One skip-gram gradient, then autograd |
| E03 · 4 min | Shifted targets, output projection, and stable cross-entropy |
| E04 · 6 min | A training loop on a compatible tiny batch |
| E05 · 4 min | Tied versus untied gradient paths |
| E06 · 4 min | Parameters, storage, and real model configurations |
| P01 · optional | PPMI from published counts |
| P02 · optional | Train and inspect toy skip-gram embeddings |
| P03 · optional | Verify a model row against its primary sources |

If you already have an older working copy, rename it before reopening the **Notebook** link; the launcher preserves existing student files. Run this notebook from top to bottom.

In [ ]:
import json
import math
from pathlib import Path

import torch
import torch.nn.functional as F

# CPU-only, reproducible small examples. Do not download a model.
torch.manual_seed(0)
V, D = 10, 4
print('PyTorch:', torch.__version__, '| vocabulary:', V, '| width:', D)

## E01 · Predict the lookup shapes

A batch of IDs has shape $(B,T)$. The table $E$ has shape $(|V|,d)$ and stores trainable parameters. Predict the shapes of `embedding.weight`, `token_ids`, `x`, and `one_hot` before running the cell.

We use contiguous toy IDs, so the number of embedding rows equals the vocabulary size. E06 will distinguish these quantities for real models.

In [ ]:
embedding = torch.nn.Embedding(V, D)
token_ids = torch.tensor([[1, 5, 5], [7, 1, 0]])
x = embedding(token_ids)
one_hot = F.one_hot(token_ids, num_classes=V).float()
e01_shapes = {'weight': tuple(embedding.weight.shape), 'ids': tuple(token_ids.shape),
              'x': tuple(x.shape), 'one_hot': tuple(one_hot.shape)}
e01_same = torch.allclose(one_hot @ embedding.weight, x)
print(e01_shapes)
print('one-hot product equals lookup:', e01_same)
print('Repeated ID 5 gives the same row:', torch.equal(x[0, 1], x[0, 2]))

**Check:** `(10, 4)`, `(2, 3)`, `(2, 3, 4)`, and `(2, 3, 10)`. The one-hot product is a mathematical explanation, not an efficient implementation. `nn.Embedding` selects rows without materializing the one-hot tensor.

The lookup for a token ID stays the same across contexts. A separate context network can transform that lookup into different contextual states.

## E02 · Predict one gradient

For one observed pair and one sampled negative pair:

$$\ell=-\log\sigma(\mathbf w^\top\mathbf c_+)-\log\sigma(-\mathbf w^\top\mathbf c_-)$$

Use $\mathbf w=(1,0.5)$, $\mathbf c_+=(0.5,1)$, $\mathbf c_-=(1,-1)$. Given $\sigma(1)=0.7311$ and $\sigma(0.5)=0.6225$, compute only $\nabla_{\mathbf c_+}\ell=(\sigma(\mathbf w^\top\mathbf c_+)-1)\mathbf w$ by hand. Then check all gradients with autograd.

The sigmoid scores observed versus sampled pairs. It is **not** a normalized next-token distribution. See [Mikolov et al. (2013), §2.2](https://papers.nips.cc/paper_files/paper/2013/file/9aa42b31882ec039965f3c4923ce901b-Paper.pdf).

In [ ]:
def sigmoid(z):
    return 1.0 / (1.0 + math.exp(-z))


def dot(a, b):
    return sum(x * y for x, y in zip(a, b))


W_VEC, C_POS, C_NEG, ETA = [1.0, 0.5], [0.5, 1.0], [1.0, -1.0], 0.5

s_pos = sigmoid(dot(W_VEC, C_POS))
s_neg = sigmoid(dot(W_VEC, C_NEG))
e02_loss = -math.log(s_pos) - math.log(1.0 - s_neg)  # sigma(-z) = 1 - sigma(z)

e02_grads = {
    "c_pos": [(s_pos - 1.0) * x for x in W_VEC],
    "c_neg": [s_neg * x for x in W_VEC],
    "w": [(s_pos - 1.0) * p + s_neg * n for p, n in zip(C_POS, C_NEG)],
}
e02_updated = {
    "c_pos": [v - ETA * g for v, g in zip(C_POS, e02_grads["c_pos"])],
    "c_neg": [v - ETA * g for v, g in zip(C_NEG, e02_grads["c_neg"])],
    "w": [v - ETA * g for v, g in zip(W_VEC, e02_grads["w"])],
}

print(f"scores: w.c_pos = {dot(W_VEC, C_POS):.1f}, w.c_neg = {dot(W_VEC, C_NEG):.1f}")
print(f"s_pos = {s_pos:.4f}, s_neg = {s_neg:.4f}, loss = {e02_loss:.4f}")
for name in ["c_pos", "c_neg", "w"]:
    grad = ", ".join(f"{g:+.4f}" for g in e02_grads[name])
    new = ", ".join(f"{v:+.4f}" for v in e02_updated[name])
    print(f"grad {name:5s} = ({grad})   updated {name:5s} = ({new})")

after = -math.log(sigmoid(dot(e02_updated["w"], e02_updated["c_pos"]))) - math.log(1.0 - sigmoid(dot(e02_updated["w"], e02_updated["c_neg"])))
print(f"loss after one step = {after:.4f}")

**Check:** the scores are $1.0$ and $0.5$, the loss is $1.2873$, and the requested gradient is $(-0.2689,-0.1345)$. With the center vector fixed, an SGD step increases its dot product with the positive context. The code above additionally updates all three vectors and checks the resulting loss.

The scalar `math.log` expressions above illustrate these small, safe numbers. For trainable tensors use the stable `F.logsigmoid` implementation below.

In [ ]:
import torch
import torch.nn.functional as F

w = torch.tensor(W_VEC, requires_grad=True)
c_pos = torch.tensor(C_POS, requires_grad=True)
c_neg = torch.tensor(C_NEG, requires_grad=True)

loss = -F.logsigmoid(w @ c_pos) - F.logsigmoid(-(w @ c_neg))
loss.backward()

e02_autograd = {"c_pos": c_pos.grad.tolist(), "c_neg": c_neg.grad.tolist(), "w": w.grad.tolist()}
print(f"loss = {loss.item():.4f}")
for name, grad in e02_autograd.items():
    print(f"autograd {name:5s} =", [round(g, 4) for g in grad], "| by hand =", [round(g, 4) for g in e02_grads[name]])

## E03 · Check the targets and loss shape

The toy sequences are `[1,2,3,4,5]` and `[5,6,7,8,9]`. Slice each sequence into inputs and next-token targets. Predict the last target in each row, the embedding/logit shapes, and the number of predictions in the mean loss.

This data is constructed for the lecture. It does not use the assignment's data or code.

In [ ]:
tokens = torch.tensor([[1, 2, 3, 4, 5], [5, 6, 7, 8, 9]])
inputs = tokens[:, :-1]
targets = tokens[:, 1:]
B, T = inputs.shape
print('inputs :', inputs.tolist())
print('targets:', targets.tolist())
print('B, T:', B, T, '| number of predictions:', B * T)
print('targets contiguous:', targets.is_contiguous())

### The model used today

`TinyLM` is a **bigram model with a learned vector bottleneck**: $h_t=E[w_t]$, then $z_t=h_tW_{out}^{\top}$. It cannot distinguish two prefixes that end in the same token ID. Next week we add a context network.

Initialize the output table to zero for an exactly uniform first prediction. Keep the input table nonzero so the output weights can learn. Initializing *both* tables to zero would stall learning through this product. The `share_weights` option is used later in E05; the main training example is untied.

In [ ]:
class TinyLM(torch.nn.Module):
    def __init__(self, vocab_size, dim, share_weights=False, zero_output=True):
        super().__init__()
        self.embedding = torch.nn.Embedding(vocab_size, dim)
        self.output = torch.nn.Linear(dim, vocab_size, bias=False)
        torch.nn.init.normal_(self.embedding.weight, std=0.02)
        if share_weights:
            self.output.weight = self.embedding.weight
        elif zero_output:
            torch.nn.init.zeros_(self.output.weight)
        else:
            torch.nn.init.normal_(self.output.weight, std=0.02)

    def forward(self, ids):
        return self.output(self.embedding(ids))


def next_token_loss(logits, labels):
    # reshape also works for the non-contiguous target slice above.
    return F.cross_entropy(logits.reshape(-1, logits.shape[-1]), labels.reshape(-1))


def count_parameters(model):
    return sum(parameter.numel() for parameter in model.parameters())


torch.manual_seed(0)
model = TinyLM(V, D)
logits = model(inputs)
loss = next_token_loss(logits, targets)
manual_loss = -F.log_softmax(logits, dim=-1).gather(-1, targets.unsqueeze(-1)).mean()
e03_shapes = {'embeddings': tuple(model.embedding(inputs).shape), 'logits': tuple(logits.shape)}
e03_uniform_loss = loss.item()
e03_manual_loss = manual_loss.item()
print(e03_shapes)
print('CE:', e03_uniform_loss, '| direct NLL:', e03_manual_loss, '| log V:', math.log(V))

**Check:** final targets $5$ and $9$, embeddings $(2,4,4)$, logits $(2,4,10)$, and eight predictions. Zero logits give loss $\log 10=2.3026$ nats and perplexity $10$.

`F.cross_entropy` takes **raw logits** and integer target IDs. It already applies log-softmax. Do not apply softmax first. Arbitrary random initialization is not guaranteed to produce this uniform baseline: its loss depends on the scale of the logits.

### Finite precision

Before running the next cell, predict what happens when float32 `sigmoid(100)` rounds to one. These two loss formulas are mathematically equivalent.

In [ ]:
stability = {}
for score in [1.0, 100.0]:
    s = torch.tensor(score, dtype=torch.float32)
    literal = -(1 - torch.sigmoid(s)).log()
    stable = -F.logsigmoid(-s)
    stability[score] = {'literal': literal.item(), 'stable': stable.item()}
    print(score, stability[score])

## E04 · Train on one tiny batch

Run 200 SGD updates on the shifted batch. Record the loss before any update and after the last update, then compare all eight predictions with the targets. Why is this not a test of generalization?

This batch is compatible with the bigram model: each observed input ID has one next-token label. A batch that assigns different next tokens to the same input cannot be fit perfectly by this model. Even fitting a compatible batch checks only the mechanics, not performance on new text.

In [ ]:
torch.manual_seed(0)
model = TinyLM(V, D)
optimizer = torch.optim.SGD(model.parameters(), lr=0.5)
e04_losses = [next_token_loss(model(inputs), targets).item()]
e04_initial_embedding = model.embedding.weight.detach().clone()
for step in range(200):
    optimizer.zero_grad()
    loss = next_token_loss(model(inputs), targets)
    loss.backward()
    optimizer.step()
    with torch.no_grad():
        e04_losses.append(next_token_loss(model(inputs), targets).item())
    if step in (0, 9, 49, 199):
        print(f'after {step + 1:3d} updates: loss = {e04_losses[-1]:.4f}')

with torch.no_grad():
    e04_predictions = model(inputs).argmax(-1)
print('initial loss:', e04_losses[0])
print('predictions:', e04_predictions.tolist())
print('targets    :', targets.tolist())

**Check:** initial loss is 2.3026 nats, final loss is below 0.1, and predictions match all eight targets. Exact losses can differ slightly across PyTorch versions. The training loop clears old gradients, runs a forward pass, computes loss, backpropagates, and takes an optimizer step.

With zero output weights, the input embedding's gradient is zero on the *first* update. The output weights change first; subsequent updates can change the input table. A nonzero gradient is not guaranteed merely because a parameter participates in the model.

The slides' loss curve samples this run at updates 0, 1, 5, 10, 20, 50, 100, 150, and 200. It is a teaching figure, not a benchmark. To evaluate generalization, reserve text that did not participate in training.

## E05 · Which rows receive gradients?

For inputs `[[1,5,5],[7,1,0]]`, which input embedding rows receive loss gradients?

Compare **two equal but independent tables** with **one shared parameter**. We initialize both cases to the same values so their logits agree. The untied output table is nonzero. The labels below illustrate gradients only; this is not the E04 memorization batch.

Weight tying must refer to the same `Parameter`, not merely copy its values once. Tie before constructing an optimizer. [Press and Wolf (2017), §3](https://aclanthology.org/E17-2025/) discusses the different updates.

In [ ]:
torch.manual_seed(7)
grad_ids = torch.tensor([[1, 5, 5], [7, 1, 0]])
grad_targets = torch.tensor([[5, 5, 2], [1, 0, 3]])
untied = TinyLM(V, D, zero_output=False)
tied = TinyLM(V, D, share_weights=True)
with torch.no_grad():
    untied.output.weight.copy_(untied.embedding.weight)
    tied.embedding.weight.copy_(untied.embedding.weight)


def active_rows(gradient):
    return (gradient.abs().sum(-1) > 0).nonzero().flatten().tolist()


e05_same_logits = torch.allclose(untied(grad_ids), tied(grad_ids))
for name, candidate in [('untied', untied), ('tied', tied)]:
    candidate.zero_grad()
    next_token_loss(candidate(grad_ids), grad_targets).backward()
    print(name, 'input-table gradient rows:', active_rows(candidate.embedding.weight.grad))

e05_untied_rows = active_rows(untied.embedding.weight.grad)
e05_tied_rows = active_rows(tied.embedding.weight.grad)
e05_gradient_sum = torch.allclose(
    tied.embedding.weight.grad,
    untied.embedding.weight.grad + untied.output.weight.grad,
    atol=1e-7,
)
print('identical logits:', e05_same_logits)
print('shared gradient = input contribution + output contribution:', e05_gradient_sum)
print('same Parameter object:', tied.output.weight is tied.embedding.weight)

**Check:** the untied input table has gradients only in rows 0, 1, 5, and 7. This example gives all ten rows nonzero gradients when tied. The shared gradient is the sum of the two pathways in the untied calculation.

At one position the output-row gradient is $(p(j)-\mathbf{1}[j=y])h$. Every row participates in softmax, including rows absent from the input. A row *can* receive a gradient; special parameter values or cancellation can still make it zero. These statements concern the loss gradient. Weight decay or previous optimizer state can change a parameter even without a new nonzero lookup gradient.

## E06 · Count parameters and storage

For $M=32{,}000$ allocated rows and $d=512$, count the input/output parameters with and without tying, and the bf16 storage for one table. No bias or context-network parameters are included. One MiB is $2^{20}$ bytes.

Then compare two official Qwen configurations. `config.vocab_size` gives the allocated table rows $M$, which can exceed the number of token IDs in the tokenizer artifact. `hidden_size` gives the width in these models. The supplied metadata contains pinned source URLs and hashes; we read it locally, without downloading weights.

In [ ]:
e06_toy_counts = {'untied': count_parameters(untied), 'tied': count_parameters(tied)}
M, width = 32_000, 512
e06_exercise = {'one_table': M * width, 'untied': 2 * M * width,
                'bf16_bytes': M * width * 2, 'bf16_mib': M * width * 2 / 2**20}
print('toy model:', e06_toy_counts)
print('exercise:', e06_exercise)

In [ ]:
# The launcher copies assets beside your working notebook. The second path
# also supports running all cells from the repository root.
asset_candidates = [Path('assets/model-configs.json'),
                    Path('slides/lecture-03/assets/model-configs.json')]
model_asset = next((path for path in asset_candidates if path.is_file()), None)
if model_asset is None:
    raise FileNotFoundError('Open this notebook with its assets directory using the course launcher.')
model_evidence = json.loads(model_asset.read_text())
e06_models = {}
for cfg in model_evidence['models']:
    one_table = cfg['embedding_rows'] * cfg['hidden_size']
    unique = one_table * (1 if cfg['tie_word_embeddings'] else 2)
    result = {'one_table': one_table, 'unique_parameters': unique, 'bf16_mib': unique * 2 / 2**20}
    e06_models[cfg['model']] = result
    print(cfg['model'], '| tokenizer IDs:', cfg['tokenizer_entries'],
          '| embedding rows:', cfg['embedding_rows'], '| tied:', cfg['tie_word_embeddings'])
    print(result)

logit_B, logit_T, logit_M = 2, 1024, 151_936
e06_logit_mib = logit_B * logit_T * logit_M * 2 / 2**20
print('materialized bf16 logits:', e06_logit_mib, 'MiB')

**Check:** one exercise table has 16,384,000 parameters and occupies 31.25 MiB in bf16. Untied input/output tables have twice as many parameters. The toy models have 80 untied parameters or 40 tied parameters.

| Model | One table | Unique input + output parameters | bf16 storage |
| --- | ---: | ---: | ---: |
| Qwen3-0.6B | 155,582,464 | 155,582,464 | 296.75 MiB |
| Qwen3-8B | 622,329,856 | 1,244,659,712 | 2,374 MiB |

Both pinned tokenizer files contain 151,669 distinct IDs (maximum 151,668), but each configuration allocates 151,936 rows. Count the union of IDs in `model.vocab` and `added_tokens`; simply adding the lengths could count the same ID twice. An embedding table must accommodate the maximum supported ID, even if an ID space has gaps.

Training also requires gradients, optimizer state, and activations. A materialized `(2,1024,151936)` bf16 logits tensor alone occupies 593.5 MiB. This is not a total-memory estimate; precision, kernels and intermediate materialization matter. Holding $B,T,d$ fixed, dense output projection takes approximately $2BTdM$ forward FLOPs, counting a multiply and an add separately. Weight tying does not remove this computation.

## P01 · PPMI from a co-occurrence table (optional)

The counts below are from Jurafsky and Martin, [Appendix J, Figure J.2](https://web.stanford.edu/~jurafsky/slp3/J.pdf). They are published word–context counts, not the invented counts in the slide illustration.

$$\mathrm{PMI}(w,c)=\log_2\frac{p(w,c)}{p(w)p(c)},\qquad \mathrm{PPMI}(w,c)=\max(\mathrm{PMI}(w,c),0)$$

Use the totals to compute PPMI(information, data), then compare a rare but associated pair such as cherry/pie. This historical background is optional; it is not needed to run a neural LM training step.

In [ ]:
import math

CONTEXTS = ["computer", "data", "result", "pie", "sugar"]
COUNTS = {
    "cherry":      [2, 8, 9, 442, 25],
    "strawberry":  [0, 0, 1, 60, 19],
    "digital":     [1670, 1683, 85, 5, 4],
    "information": [3325, 3982, 378, 5, 13],
}

TOTAL = sum(sum(row) for row in COUNTS.values())
ROW_TOTAL = {word: sum(row) for word, row in COUNTS.items()}
COLUMN_TOTAL = {c: sum(COUNTS[w][j] for w in COUNTS) for j, c in enumerate(CONTEXTS)}


def pmi(word, context):
    # Pointwise mutual information in bits; -inf when the pair never co-occurs.
    joint = COUNTS[word][CONTEXTS.index(context)] / TOTAL
    if joint == 0:
        return float("-inf")
    return math.log2(joint / ((ROW_TOTAL[word] / TOTAL) * (COLUMN_TOTAL[context] / TOTAL)))


def ppmi(word, context):
    return max(pmi(word, context), 0.0)


print("grand total:", TOTAL, "| row totals:", ROW_TOTAL, "| column totals:", COLUMN_TOTAL)
for word, context in [("information", "data"), ("cherry", "pie"), ("strawberry", "computer"), ("digital", "pie")]:
    print(f"PMI({word}, {context}) = {pmi(word, context):8.4f}   PPMI = {ppmi(word, context):.4f}")

**Check:** $p(\text{information},\text{data}) = 3982/11716 = 0.3399$, $p(\text{information}) = 7703/11716 = 0.6575$, $p(\text{data}) = 5673/11716 = 0.4842$, so $\mathrm{PMI} = \log_2(0.3399/(0.6575 \times 0.4842)) = 0.0944$ bits. The pair is frequent, yet barely more frequent than chance, because both words are frequent. `cherry` and `pie` are rarer but strongly associated ($4.38$ bits). A zero count gives $-\infty$, and a negative PMI needs a large corpus to be reliable: PPMI sets both to $0$.

## P02 · Train and inspect toy skip-gram vectors (optional)

Two tables score observed and sampled pairs, as in E02. This larger toy uses window radius 3, five negative samples per pair, and a noise distribution proportional to frequency raised to 0.75. The corpus is generated from templates with deliberately distinct contexts for people, animals, foods, and places. These patterns are designed examples, not findings about language.

This practice takes a few seconds on CPU. It is outside the class timing. Neighbors and analogy results on this small corpus can be fragile; interpret them as a way to inspect the learned representation.

In [ ]:
import random


def build_toy_corpus(sentences_per_template=150, seed=2026):
    rng = random.Random(seed)
    people = {
        "king": ("he", "rules", "palace"), "queen": ("she", "rules", "palace"),
        "prince": ("he", "rules", "palace"), "princess": ("she", "rules", "palace"),
        "man": ("he", "works", "village"), "woman": ("she", "works", "village"),
        "boy": ("he", "works", "village"), "girl": ("she", "works", "village"),
    }
    animals, foods = ["cat", "dog", "horse", "bird"], ["rice", "bread", "fish", "apple"]
    capitals = {"beijing": "china", "paris": "france", "london": "england", "tokyo": "japan"}
    sentences = []
    for _ in range(sentences_per_template):
        person = rng.choice(list(people))
        pronoun, verb, place = people[person]
        age = "young " if person in ("prince", "princess", "boy", "girl") else ""
        sentences.append(f"the {age}{person} said {pronoun} {verb} in the {place}")
        sentences.append(f"the {age}{person} eats {rng.choice(foods)} in the {place}")
        sentences.append(f"the {rng.choice(animals)} chases the {rng.choice(animals)} and eats {rng.choice(foods)}")
        capital = rng.choice(list(capitals))
        sentences.append(f"{capital} is the capital city of {capitals[capital]}")
        sentences.append(f"the {person} travels from the {place} to {rng.choice(list(capitals))}")
    return [s.split() for s in sentences]


TOY_SENTENCES = build_toy_corpus()
VOCAB = sorted({word for sentence in TOY_SENTENCES for word in sentence})
WORD_TO_ID = {word: i for i, word in enumerate(VOCAB)}
print(len(TOY_SENTENCES), "sentences,", sum(map(len, TOY_SENTENCES)), "tokens,", len(VOCAB), "word types")
print(" ".join(TOY_SENTENCES[0]))

In [ ]:
def positive_pairs(sentences, window=3):
    pairs = []
    for sentence in sentences:
        ids = [WORD_TO_ID[word] for word in sentence]
        for i, center in enumerate(ids):
            for j in range(max(0, i - window), min(len(ids), i + window + 1)):
                if j != i:
                    pairs.append((center, ids[j]))
    return torch.tensor(pairs)


PAIRS = positive_pairs(TOY_SENTENCES)
counts = torch.bincount(torch.tensor([WORD_TO_ID[w] for s in TOY_SENTENCES for w in s]), minlength=len(VOCAB)).float()
NOISE = counts.pow(0.75) / counts.pow(0.75).sum()       # P(w) proportional to count^0.75
print("positive pairs:", tuple(PAIRS.shape), "| first pairs:", [(VOCAB[a], VOCAB[b]) for a, b in PAIRS[:4].tolist()])


class SkipGram(torch.nn.Module):
    def __init__(self, vocab_size, dim):
        super().__init__()
        self.W = torch.nn.Embedding(vocab_size, dim)     # center-word vectors
        self.C = torch.nn.Embedding(vocab_size, dim)     # context-word vectors
        torch.nn.init.uniform_(self.W.weight, -0.5 / dim, 0.5 / dim)   # word2vec's initialization:
        torch.nn.init.zeros_(self.C.weight)                            # small W, zero C

    def forward(self, center, positive, negatives):      # (N,), (N,), (N, k)
        w = self.W(center)                               # (N, d)
        positive_score = (w * self.C(positive)).sum(-1)                       # (N,)
        negative_score = torch.einsum("nd,nkd->nk", w, self.C(negatives))     # (N, k)
        return -(F.logsigmoid(positive_score) + F.logsigmoid(-negative_score).sum(-1)).mean()


def train_skipgram(dim=16, k=5, epochs=15, batch_size=256, lr=0.01, seed=0):
    generator = torch.Generator().manual_seed(seed)
    torch.manual_seed(seed)
    model = SkipGram(len(VOCAB), dim)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = []
    for epoch in range(epochs):
        order = torch.randperm(len(PAIRS), generator=generator)
        total = 0.0
        for start in range(0, len(PAIRS), batch_size):
            batch = PAIRS[order[start:start + batch_size]]
            negatives = torch.multinomial(NOISE, len(batch) * k, replacement=True, generator=generator).view(len(batch), k)
            loss = model(batch[:, 0], batch[:, 1], negatives)
            if epoch == 0 and start == 0:
                print(f"first minibatch: loss = {loss.item():.4f}")
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total += loss.item() * len(batch)
        history.append(total / len(PAIRS))
        print(f"epoch {epoch + 1}: mean loss = {history[-1]:.4f}")
    return model, history


skipgram, p02_history = train_skipgram()

With five negatives and a zero context table, the first minibatch has loss $6\log 2=4.1589$. Positive and noise distributions overlap, so do not expect every sampled binary classification to be perfectly separable.

### Inspect the vectors

Normalize nonzero vectors and use their dot product as cosine similarity. The analogy query below is a diagnostic, not a language-understanding evaluation.

In [ ]:
VECTORS = F.normalize(skipgram.W.weight.detach(), dim=1)        # unit length, so a dot product is a cosine


def nearest(word, k=3):
    scores = VECTORS @ VECTORS[WORD_TO_ID[word]]
    scores[WORD_TO_ID[word]] = -1.0
    return [(VOCAB[i], round(scores[i].item(), 2)) for i in scores.topk(k).indices.tolist()]


def analogy(a, a_star, b, k=3):
    # a is to a_star as b is to ?
    query = F.normalize(VECTORS[WORD_TO_ID[a_star]] - VECTORS[WORD_TO_ID[a]] + VECTORS[WORD_TO_ID[b]], dim=0)
    scores = VECTORS @ query
    for word in (a, a_star, b):
        scores[WORD_TO_ID[word]] = -1.0
    return [VOCAB[i] for i in scores.topk(k).indices.tolist()]


for word in ["king", "cat", "rice", "paris"]:
    print(f"{word:6s} ->", nearest(word))
print("man : woman = king : ?   ", analogy("man", "woman", "king"))
print("man : woman = prince : ? ", analogy("man", "woman", "prince"))

The templates give groups distinguishable contexts, so recovering these groups is an intended result of the data construction. Try changing the seed or dimension. Which neighbors stay stable? For word types with identical context distributions, this objective has no contextual evidence to distinguish their meanings.

## P03 · Verify one model row (optional course contribution)

Work in pairs on the [model-table issue #154](https://github.com/baojian/llm-26-fall/issues/154), then ask another pair to check your evidence and arithmetic. Follow that issue's scope for release dates and context lengths. Keep **Sources and Notes** as one column. This is an optional contribution to the course, not a new grading requirement.

The next cell prints the pinned official sources already recorded for E06. Reproducing the source check requires network access *outside this offline notebook run*: download the linked `config.json` and `tokenizer.json` only, verify their SHA-256 hashes, and inspect the fields below. Model weights are unnecessary. Do not infer unavailable artifacts for closed models.

1. Read `vocab_size`, `hidden_size`, and `tie_word_embeddings` from the configuration.
2. Count the union of IDs in `tokenizer['model']['vocab'].values()` and `entry['id']` for `entry` in `tokenizer['added_tokens']`.
3. Record the maximum ID and check that it fits the table.
4. Compute unique table parameters and bytes; state the dtype and whether weights are tied.
5. Include the model repository, immutable revision, source URLs, and any distinction between reported and derived values.

The counting method above applies to these JSON BPE tokenizers. Other tokenizer formats require their own inspection; do not assume every model publishes the same file layout.

In [ ]:
for cfg in model_evidence['models']:
    print(cfg['model'], '| revision:', cfg['revision'])
    for kind, evidence in cfg['sources'].items():
        print(kind, evidence['url'])
        print('SHA-256:', evidence['sha256'])

### Further reading

- [CS336 Lecture 2](https://github.com/stanford-cs336/lectures/blob/main/lecture_02.py): tensors, memory, gradients, and the training loop.
- [PyTorch CrossEntropyLoss](https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html): raw logits and integer targets.
- [Bengio et al. (2003), Section 2](https://www.jmlr.org/papers/v3/bengio03a.html): a context network over learned word vectors.
- [Press and Wolf (2017)](https://aclanthology.org/E17-2025/): tied input/output embeddings.
- [Qwen3-Embedding report](https://arxiv.org/abs/2506.05176): representations trained for retrieval, distinct from raw token lookups.

The course deck links to `classical-reading.md` for optional background on counting and static embeddings.